# N065 · 回溯去重与剪枝

**目标：** 区分同层重复和同一路径重复使用。

**先修：** N064, N037。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 排序；起点；used标记；正数剪枝前提；状态恢复。

**配套讲解来源：** [同名逐章意见](../../comment/065_backtracking_dedup_pruning.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章（N064）我们默认输入里没有重复元素。这一章处理两个新麻烦：

**麻烦一：值重复了，答案会重复。** 比如输入 `[1,2,2]` 求子集：两个 2 是不同的数组下标，但值一模一样。如果沿用上一章的做法，`[1,2]`（取第 1 个 2）和 `[1,2]`（取第 2 个 2）会被当成两个答案输出，但用户眼里它们就是同一个子集。正确答案应该是 6 个：`[]`、`[1]`、`[2]`、`[1,2]`、`[2,2]`、`[1,2,2]`——而不是 2³=8 个。同理 `[1,1,2]` 的排列只有 3 种：`[1,1,2]`、`[1,2,1]`、`[2,1,1]`，而不是 3!=6 种。

**麻烦二：组合和允许同一个数反复用。** 比如 candidates=`[2,3]`、target=7，问有哪些选法让和恰好等于 7：2+2+3=7，所以答案是 `[2,2,3]` 一种（2 用了两次！）。再比如 “自动测试”部分 里的例子 `[2,3,6,7]`、target=7，答案是 `{(2,2,3),(7,)}` 两种：三个数加起来 7，或者一个 7 直接达标。"可重复使用"意味着同一个候选可以连续选中多次。

一个具体到数字的例子：输入 `nums=[1,2,2]`，`subsets_with_dup` 输出 6 个子集（上面列过），`permute_unique` 输出 3 个排列，`combination_sum([1,2],5)` 输出 `[[1,1,1,1,1],[1,1,1,2],[1,2,2]]`——每个候选可以用任意多次，但每种组合只保留升序的那一个。

## 2. 基础知识：先读懂这些词

- **同层去重**：回溯树里，同一个父节点下的兄弟分支叫"同一层"。如果同一层里两个分支选的值相同（比如两个 2），它们后面能做的事完全一样，输出也会完全一样，所以只走第一个、跳过后面的。这是消除"值重复导致答案重复"的核心手段。
- **排序（sorted）**：去重的前提。把输入排好序后，相同的值一定挨在一起，于是"和前一个一样"只需比较 `a[i]==a[i-1]` 这一对邻居，不用建哈希表。
- **起点（start）**：组合类问题里"本层允许从哪个下标开始选"的参数，保证答案按升序生成，避免顺序性重复。
- **used 标记**：排列题里记录"哪个下标已在路径上"的布尔数组。去重版排列还要配合"相同值按固定下标顺序使用"的规则，即第 2 个 2 必须等第 1 个 2 用了之后才能用。
- **剪枝（pruning）**：提前砍掉"必然失败"的分支。本章的 `if a[i]>remaining: break` 就是一刀：候选已经排好序，当前的最小候选都比剩余目标大，后面的只会更大，整层直接放弃。
- **正数前提**：组合和要求所有候选必须严格大于 0。原因：每选一个数 remaining 就要严格变小，递归才会在有限步内到 0 或变负；如果有 0 或负数，remaining 可能永远不减甚至变大，递归就停不下来了。
- **状态恢复（undo）**：和上一章一样，递归回来后 pop 路径、还原 used，让兄弟分支在干净状态下工作。

## 3. 思路推导：从小例子开始

**Step 1：先排序。** 三个函数的第一行都是 `a=sorted(...)`。排序花了 O(n log n)，换来的是"相同值相邻"这个宝贵性质，后面所有去重判断都靠它。

**Step 2：子集去重——同层跳过相同值。** 关键代码是 `if i>start and a[i]==a[i-1]: continue`。它的意思是：在当前这层的循环里（i>start 保证 i-1 是同层循环里刚试过的兄弟，不是路径上的父亲），如果这个值和刚试过的值一样，就跳过——刚才那个分支已经把"选一个这个值"的所有后续都覆盖了。

**Step 3：排列去重——相同值按顺序用。** 判断条件是 `i>0 and x==a[i-1] and not used[i-1]`：当前数和前一个数值相同，而且前一个相同的数还没被用，就跳过。换成人话："两个 1 排队，只允许下标小的 1 先进队。"这样 `[1,1,2]` 的 `1a,1b` 和 `1b,1a` 只有前者被生成。

**Step 4：组合和——允许重复使用候选。** 递归调用是 `visit(i, remaining-a[i])`，注意传的是 `i` 而不是 `i+1`：下一层还可以从同一个下标再选一次，这就实现了"可重复"。同时候选先 `sorted(set(candidates))` 去重排序，让答案天然升序。

**Step 5：剪枝与终止。** 候选升序后，一旦 `a[i]>remaining` 就 `break` 掉整层；候选全为正保证 remaining 严格下降，递归必停。

**手算演示 1：`subsets_with_dup([1,2,2])`（对应 “运行示例”部分 表格第一行）。** 排序后 a=[1,2,2]。

1. `visit(0)`：先收集当前 path，即 `[]`。循环 i=0,1,2。
2. i=0 选 1 → `visit(1)`（start=1）收集 `[1]`；其内层 i=1 选第 1 个 2 → `visit(2)`（start=2）收集 `[1,2]`；这一内层的循环只有 i=2 一个候选，条件 `i>start` 即 2>2 不成立，所以第 2 个 2 **不算**同层重复、照选 → `visit(3)` 收集 `[1,2,2]`。（关键点：这两个 2 出现在不同深度——跨层使用同值是允许的，这就是 `[2,2]` 合法的原因。）
3. 回到 `visit(1)` 层（path=[1] 的下一层），i=2：`i>start`（2>1）且 a[2]==a[1]，**跳过**——因为"选第 2 个 2"和刚做过的"选第 1 个 2"后续完全一样。
4. 回到根层，i=1 选 2 → 收集 `[2]` → 内层再选 2 → `[2,2]`；根层 i=2 与 i=1 同值同层，跳过。
5. 总计 6 个：`[]`、`[1]`、`[1,2]`、`[1,2,2]`、`[2]`、`[2,2]`。和 “运行示例”部分 表格一致。

**手算演示 2：`combination_sum([1,2],5)`（“运行示例”部分 表格第三行）。** a=[1,2]，target=5。

1. `visit(0,5)`：i=0 选 1 → `visit(0,4)`（还是从下标 0 开始，可重复）→ 一路 1 下去 → `[1,1,1,1,1]`（remaining 到 0 收集）。
2. 在 remaining=3 那层改选 2 → `[1,1,1,2]`；在 remaining=2 那层……实际展开顺序：每层先试 1 再试 2，最终三条答案 `[1,1,1,1,1]`、`[1,1,1,2]`、`[1,2,2]`。
3. 验证：5=1×5=1×3+2=1+2+2，恰好三种，没有第四种。

**手算演示 3：`permute_unique([1,1,2])`。** a=[1,1,2]。第一层 i=0 选第 1 个 1；第二层 i=1（第 2 个 1，used[0]=True 所以 `not used[i-1]` 为假，不跳过）→ 第三层选 2 → `[1,1,2]`。若第一层直接想选 i=1 的 1：a[1]==a[0] 且 used[0]=False → 跳过。三个答案如第一节所列。

最后补一张"同层 vs 跨层"的对照图，这一对概念是全章的命门：

```
根层循环:  选1   选2a   选2b   ← 选2b 与选2a 同层同值，跳过！
                       │
                       └─ 但在"选2a"分支的下一层里再选2b 是允许的 → 生成 [2,2]
```

同层跳过挡的是"兄弟分支撞车"（生成两个一模一样的答案）；跨层放行允许的是"路径里连用两个 2"（这是合法答案）。`i>start` 这一个比较就是两种情况的分界线。

## 4. 核心代码：subsets_with_dup

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def subsets_with_dup(nums):
    a = sorted(nums)
    out = []
    path = []

    def visit(start):
        out.append(path[:])
        for i in range(start, len(a)):
            if i > start and a[i] == a[i - 1]:
                continue
            path.append(a[i])
            visit(i + 1)
            path.pop()
    visit(0)
    return out
```

### 逐段读懂代码

### 4.1 `subsets_with_dup(nums)`

```python
def subsets_with_dup(nums):
    a = sorted(nums)
    out = []
    path = []

    def visit(start):
        out.append(path[:])
        for i in range(start, len(a)):
            if i > start and a[i] == a[i - 1]:
                continue
            path.append(a[i])
            visit(i + 1)
            path.pop()
    visit(0)
    return out
```

- `a=sorted(nums)`：先排序，让相同值相邻；从此只用 a，不动原输入。
- `out.append(path[:])`：注意这里**一进来就收集**，不像上一章要走到末尾。因为子集的"答案"是每个节点（不只是叶子）——空路径 `[]` 也是子集，所以每个节点都要存一份拷贝。
- `if i>start and a[i]==a[i-1]: continue`：全函数最关键的一行。`i>start` 区分了两种"前一个"：如果 i-1≥start，说明 a[i-1] 是**同一层**刚试过的兄弟，同值就跳过；如果 i==start，a[i-1] 是**路径上的父亲**层用过的，这种情况不跳过（正是它允许 `[2,2]` 出现）。
- `path.append(a[i]); visit(i+1); path.pop()`：标准三部曲；子集这里每个元素最多用一次，所以下一层从 `i+1` 开始。

### 4.2 `permute_unique(nums)`

```python
def permute_unique(nums):
    a = sorted(nums)
    used = [False] * len(a)
    out = []
    path = []

    def visit():
        if len(path) == len(a):
            out.append(path[:])
            return
        for i, x in enumerate(a):
            if used[i] or (i > 0 and x == a[i - 1] and (not used[i - 1])):
                continue
            used[i] = True
            path.append(x)
            visit()
            path.pop()
            used[i] = False
    visit()
    return out
```

- `if used[i] or (i>0 and x==a[i-1] and not used[i-1]): continue`：两个跳过条件用 or 连接。第一个是"这个下标已在路径上"；第二个读作"我和前一个邻居值相同、且那个邻居还没被用"——即"轮到我之前必须先用掉前面那个相同的值"。这条规矩把相同值的多种使用顺序压成了一种。
- `used[i]=True; path.append(x); visit(); path.pop(); used[i]=False`：choose/explore/undo 一行排开，撤销顺序与做选择完全对称。

### 4.3 `combination_sum(candidates,target)`

```python
def combination_sum(candidates, target):
    a = sorted(set(candidates))
    if any((x <= 0 for x in a)) or target < 0:
        raise ValueError('positive candidates and nonnegative target required')
    out = []
    path = []

    def visit(start, remaining):
        if remaining == 0:
            out.append(path[:])
            return
        for i in range(start, len(a)):
            if a[i] > remaining:
                break
            path.append(a[i])
            visit(i, remaining - a[i])
            path.pop()
    visit(0, target)
    return out
```

- `a=sorted(set(candidates))`：先用 set 把重复候选去掉（两个 2 当成一个 2 用，反正可以重复选），再排序。
- `if any(x<=0 for x in a) or target<0: raise ValueError(...)`：输入契约——候选必须全为正、目标必须非负。这不只是防呆：如果允许 0 或负候选，remaining 不会严格下降，递归可能永远不终止，程序会栈溢出而不是给出答案。
- `if remaining==0: out.append(path[:])`：剩余目标恰好烧到 0，当前 path 就是一条答案。remaining 只会变小不会变负吗？会变负的分支被下一行挡住了。
- `if a[i]>remaining: break`：剪枝。候选升序，最小的一个都已经超过 remaining，后面更大，整层没有戏；而且注意是 `break` 不是 `continue`，因为后面的候选只会更超标，逐个试是浪费时间。
- `visit(i,remaining-a[i])`：下一层的 start 传 `i`（不是 i+1），意思是"同一个候选还能再用"，这就是"可重复使用"的实现点。

## 5. 分段实现：按顺序运行

**本章接口：** `subsets_with_dup(nums)`、`permute_unique(nums)`、`combination_sum(candidates, target)`

### subsets_with_dup

In [1]:
def subsets_with_dup(nums):
    a = sorted(nums)
    out = []
    path = []

    def visit(start):
        out.append(path[:])
        for i in range(start, len(a)):
            if i > start and a[i] == a[i - 1]:
                continue
            path.append(a[i])
            visit(i + 1)
            path.pop()
    visit(0)
    return out

### permute_unique

In [2]:
def permute_unique(nums):
    a = sorted(nums)
    used = [False] * len(a)
    out = []
    path = []

    def visit():
        if len(path) == len(a):
            out.append(path[:])
            return
        for i, x in enumerate(a):
            if used[i] or (i > 0 and x == a[i - 1] and (not used[i - 1])):
                continue
            used[i] = True
            path.append(x)
            visit()
            path.pop()
            used[i] = False
    visit()
    return out

### combination_sum

In [3]:
def combination_sum(candidates, target):
    a = sorted(set(candidates))
    if any((x <= 0 for x in a)) or target < 0:
        raise ValueError('positive candidates and nonnegative target required')
    out = []
    path = []

    def visit(start, remaining):
        if remaining == 0:
            out.append(path[:])
            return
        for i in range(start, len(a)):
            if a[i] > remaining:
                break
            path.append(a[i])
            visit(i, remaining - a[i])
            path.pop()
    visit(0, target)
    return out

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,2]
show_table(['对象','去重答案'],[('子集',subsets_with_dup(a)),('排列',permute_unique(a)),('组合和 target=5',combination_sum([1,2],5))])

对象,去重答案
子集,"[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]"
排列,"[[1, 2, 2], [2, 1, 2], [2, 2, 1]]"
组合和 target=5,"[[1, 1, 1, 1, 1], [1, 1, 1, 2], [1, 2, 2]]"


## 7. 正确性、适用条件与复杂度

同层相同值的分支在剩余可选值上等价，保留一个代表不会漏掉不同值答案；不同深度可继续使用同一值。正候选使remaining严格下降，保证终止。

**代价：** 输出敏感的指数算法；排序O(n log n)，工作空间为最大递归深度，组合和深度最多target/min(candidates)。

### 展开理解

**为什么同层跳过不会漏答案？** 想象同一层有两个分支都选了值 2（分别是下标 1 和 2 的 2）。它们进入下一层后，眼前"还能选哪些值"的处境完全一样（都是从剩下的候选里选），所以两个分支能生成的所有答案一模一样。跳过第二个分支，只是不把同样的答案再打印一遍，不会漏掉任何"长得不一样"的答案。而不同深度继续用同一个值（比如 `[2,2]`）没有被跳过，因为那不是同层重复，是"再选一个 2"的合法行为。

**排列为什么"按顺序使用相同值"就够了？** 两个相同值的任意使用顺序最终生成的排列字符串没有区别，我们只需保证其中一种顺序（按下标从小到大）被枚举。条件 `not used[i-1]` 正是在强制这个顺序。

**组合和为什么一定会停？** 每次递归 remaining 至少减掉最小的正候选（≥1），所以 remaining 像倒计时一样严格下降，最多 target/min(candidates) 层就归零或被剪枝拦住。

**复杂度（结合数字说）：** 这类算法的耗时主要由"最终输出的答案条数"决定，是输出敏感的指数级。比如 `combination_sum([1],30)` 恰好只有 1 条答案但递归 30 层；而 `combination_sum([1,2],30)` 的答案条数随 target 增长很快。排序只花 O(n log n)，工作栈深度最多 target/min(candidates)（候选都取最小值 1、target=30 时深 30 层）。所以这类题的 target、n 都不会给很大。

**数一数去重到底省了多少：** 以 [1,2,2] 求子集为例，不去重的选择树有 2³=8 个叶子，同层跳过后只剩 6 个（少掉的正是那两个"换一个 2"的重复答案）；排列更夸张，[1,1,2] 不去重有 3!=6 个叶子，去重后只剩 3 个。重复值越多砍得越狠：把输入换成 [2,2,2]，排列从 6 个砍到 1 个。排序花掉的 O(n log n) 换来这样的裁剪，非常划算。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 逐条解释：

- `assert len(subsets_with_dup([1,2,2]))==6`：正常值。两个 2 只能算一个，6 这个数字直接验证同层去重生效（不去重会是 2³=8）。
- `assert len(permute_unique([1,1,2]))==3`：正常值。3 也恰是公式 3!/2!=3，验证排列的值去重。
- `assert combination_sum([2,3],0)==[[]]`：边界值。目标为 0 时唯一答案是"什么都不选"，即一个包含空列表的列表；同时它验证 remaining==0 的收集分支在最顶端就命中。
- `for a in product([1,2],repeat=5): ...`：这是**穷举交叉验证**。`product([1,2],repeat=5)` 生成全部 32 个长度为 5 的 1/2 数组；对每个数组，用 itertools 的 combinations（按每种长度枚举再排序去重转元祖放入集合）当参考答案，和我们子集函数的输出集合比对，同时检查输出条数等于参考集合大小（没有内部重复）。排列部分用 `set(permutations(a))` 做同样的比对。32 个输入全过，说明去重逻辑在各种重复分布下都对。
- `assert {tuple(x) for x in combination_sum([2,3,6,7],7)}=={(2,2,3),(7,)}`：LeetCode 39 的经典例子。7 可以是 2+2+3，也可以是单独一个 7；注意 2+2+2=6 不行、2+3+3=8 超了、6+? 不行，所以恰好两个答案。

In [5]:
from itertools import combinations, permutations, product

assert len(subsets_with_dup([1, 2, 2])) == 6

assert len(permute_unique([1, 1, 2])) == 3

assert combination_sum([2, 3], 0) == [[]]

for a in product([1, 2], repeat=5):
    out = subsets_with_dup(a)
    ref = {tuple(sorted(t)) for k in range(6) for t in combinations(a, k)}
    assert set(map(tuple, out)) == ref and len(out) == len(ref)
    out = permute_unique(a)
    ref = set(permutations(a))
    assert set(map(tuple, out)) == ref and len(out) == len(ref)

assert {tuple(x) for x in combination_sum([2, 3, 6, 7], 7)} == {(2, 2, 3), (7,)}

print('N065: 所有本章断言通过')

N065: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释有负数时按和超标剪枝可能失效。

**练习 2：** 比较可重复与不可重复选取。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（解释有负数时按和超标剪枝可能失效）**：提示——剪枝 `a[i]>remaining: break` 依赖"加上去只会让和更大"这个直觉，而这个直觉只在候选全为正时成立。手算示例：candidates=[-2,3]、target=1，选了 3 之后 remaining=-2，如果按"超过就 break"的逻辑这一支似乎死了，但再选一个 -2 恰好凑回 1（答案是 [3,-2]）；更糟的是候选含 0 时 [0,3]、target=3 会无限选 0 停不下来。你可以构造一个这样的输入，观察 ValueError 的防护，再写一个允许负数的版本说明需要什么额外机制（比如限制选的个数）。
- **练习 2（比较可重复与不可重复选取）**：提示——本章 `combination_sum` 下一层传 `i`（可重复），对照上一章 `combine` 传 `x+1`（不可重复）。把本章函数里的 `visit(i,...)` 改成 `visit(i+1,...)` 再跑 `combination_sum([2,3,6,7],7)`，答案会从两个变成一个：`[2,2,3]` 消失了（它需要 2 用两次），只剩 `(7,)`，一行改动直接让你看见"可重复"的实现点在哪。再想想 LeetCode 40（candidates 有重复且每个只用一次）需要同时用上"同层去重 + i+1"两件工具，这个对比就是这道练习的落点。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def subsets_with_dup(nums):
    a = sorted(nums)
    out = []
    path = []

    def visit(start):
        out.append(path[:])
        for i in range(start, len(a)):
            if i > start and a[i] == a[i - 1]:
                continue
            path.append(a[i])
            visit(i + 1)
            path.pop()
    visit(0)
    return out

def permute_unique(nums):
    a = sorted(nums)
    used = [False] * len(a)
    out = []
    path = []

    def visit():
        if len(path) == len(a):
            out.append(path[:])
            return
        for i, x in enumerate(a):
            if used[i] or (i > 0 and x == a[i - 1] and (not used[i - 1])):
                continue
            used[i] = True
            path.append(x)
            visit()
            path.pop()
            used[i] = False
    visit()
    return out

def combination_sum(candidates, target):
    a = sorted(set(candidates))
    if any((x <= 0 for x in a)) or target < 0:
        raise ValueError('positive candidates and nonnegative target required')
    out = []
    path = []

    def visit(start, remaining):
        if remaining == 0:
            out.append(path[:])
            return
        for i in range(start, len(a)):
            if a[i] > remaining:
                break
            path.append(a[i])
            visit(i, remaining - a[i])
            path.pop()
    visit(0, target)
    return out

# 示例与回归测试
from itertools import combinations, permutations, product

assert len(subsets_with_dup([1, 2, 2])) == 6

assert len(permute_unique([1, 1, 2])) == 3

assert combination_sum([2, 3], 0) == [[]]

for a in product([1, 2], repeat=5):
    out = subsets_with_dup(a)
    ref = {tuple(sorted(t)) for k in range(6) for t in combinations(a, k)}
    assert set(map(tuple, out)) == ref and len(out) == len(ref)
    out = permute_unique(a)
    ref = set(permutations(a))
    assert set(map(tuple, out)) == ref and len(out) == len(ref)

assert {tuple(x) for x in combination_sum([2, 3, 6, 7], 7)} == {(2, 2, 3), (7,)}

print('N065: 所有本章断言通过')

N065: 所有本章断言通过


## 11. 代表题与迁移

- **90. Subsets II**：对应 `subsets_with_dup`，练的是"排序 + 同层跳相同值"这套去重组合拳。
- **47. Permutations II**：对应 `permute_unique`，练的是"相同值按下标顺序使用"的排列去重条件。
- **39. Combination Sum**：对应 `combination_sum`，练的是"下一层 start 传 i 实现可重复选取 + 正数前提保证终止"。
- **40. Combination Sum II**：这题是 39 加上"每个候选只用一次且候选有重复"，需要把本章的同层去重（子集那招）嫁接到组合和框架上，正好是练习 2 的延伸。

一句话总结本章：**排序让相同值相邻，`i>start` 区分同层与跨层，同层跳过保去重、跨层放行保 [2,2]，正数前提保终止，`break` 剪枝保速度。**

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。